In [ ]:
import itertools
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
eval_tmin, eval_tmax = 0.3, 0.5
pval_threshold = 1e-2

plot_n_erps = 20

outdir = "."

In [ ]:
all_epoch_paths = list(Path("epochs").glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epochs", str(path))[0]
    if subject_name == "EC282":
        # missing ecog data
        continue
    epochs[subject_name] = mne.read_epochs(str(path)).pick("ecog")

In [ ]:
ttest_results = {}

In [ ]:
for subject, subject_epochs in tqdm(epochs.items()):
    md = subject_epochs.metadata
    print(subject)

    # does the acoustic input bias toward the first phoneme or the second phoneme?
    md["left_bias"] = md.resampled <= 3
    # what is the bias of the acoustic input in phoneme category?
    md["initial_phoneme"] = np.where(md.resampled <= 3, md.phoneme_pair.str[0], md.phoneme_pair.str[1])
    md["target_phoneme"] = md.word_end.str[0]
    # is the acoustic input compatible with the eventual lexical evidence or not?
    md["mismatch"] = md.initial_phoneme != md.target_phoneme

    epochs_grouped = {}
    for (phoneme_pair, initial_phoneme, mismatch), rows in md.groupby(["phoneme_pair", "initial_phoneme", "mismatch"]):
        # TODO why are there sometimes more metadata elements than epochs?
        index_rows = [index for index in rows.index if index < len(subject_epochs)]
        epochs_grouped[(phoneme_pair, initial_phoneme, mismatch)] = subject_epochs[index_rows]

    for (phoneme_pair, initial_phoneme), _ in md.groupby(["phoneme_pair", "initial_phoneme"]):
        mismatch_evoked = epochs_grouped[(phoneme_pair, initial_phoneme, True)].copy() \
            .crop(eval_tmin, eval_tmax) \
            .get_data().mean(axis=2).T
        match_evoked = epochs_grouped[(phoneme_pair, initial_phoneme, False)].copy() \
            .crop(eval_tmin, eval_tmax) \
            .get_data().mean(axis=2).T
        # ^ these are n_channels * n_epochs

        for channel in range(mismatch_evoked.shape[0]):
            ttest_results[subject, phoneme_pair, initial_phoneme, channel] = pd.Series(
                dict(zip(["t", "p"], ttest_ind(mismatch_evoked[channel], match_evoked[channel]))))
    

In [ ]:
ttest_results_df = pd.concat(ttest_results, names=["subject", "phoneme_pair", "initial_phoneme", "channel"]).unstack().sort_values("p")
# bonferroni correction
ttest_results_df["p"] *= len(ttest_results_df)
ttest_results_df.head(50)

In [ ]:
ttest_results_df.to_csv(Path(outdir) / "ttest_results.csv")

In [ ]:
ttest_overall = ttest_results_df[ttest_results_df.p < pval_threshold]
ttest_overall = (ttest_overall.t.unstack(["subject", "channel"]).dropna(axis=1, how="all"))
sns.clustermap(ttest_overall.fillna(0.))

In [ ]:
ttest_results_df.groupby(["subject", "channel", "phoneme_pair"]).p.min().sort_values().head(10)

In [ ]:
plot_keys = ttest_results_df.groupby(["subject", "channel", "phoneme_pair"]).p.min().sort_values().head(plot_n_erps).index
plot_epochs = {}
for subject, channel, phoneme_pair in tqdm(plot_keys, desc="Preparing plot data"):
    # n_epochs * 1 * n_times
    plot_epochs_i = epochs[subject][f"phoneme_pair == '{phoneme_pair}'"].copy().pick([channel])
    plot_epochs[subject, channel, phoneme_pair] = pd.merge(
        plot_epochs_i.to_data_frame().rename(columns={plot_epochs_i.ch_names[0]: "value"}),
        plot_epochs_i.metadata[["initial_phoneme", "mismatch"]],
        left_on="epoch", right_index=True)
    
plot_epochs = pd.concat(plot_epochs, names=["subject", "channel", "phoneme_pair"]).reset_index()
plot_epochs["channel_label"] = plot_epochs.subject.str.cat(plot_epochs.channel.astype(str), sep="_")

In [ ]:
plot_epochs

In [ ]:
g = sns.FacetGrid(data=plot_epochs, col="channel_label", col_wrap=2, aspect=2)
hue_order = list(itertools.chain.from_iterable(plot_epochs.phoneme_pair.unique()))
def f(data, **kwargs):
    ax = plt.gca()
    sns.lineplot(data=data, x="time", y="value",
                 hue="initial_phoneme", hue_order=hue_order,
                 style="mismatch", style_order=[False, True],
                 ax=ax)
g.map_dataframe(f)
for ax in g.axes.flat:
    ax.legend()

In [ ]:
g.savefig(Path(outdir) / "erps.pdf")